---
description: Condition structured OCR Markdown for DOM semantics analysis
output-file: preprocessing.conditioning.markdown.html
title: Semantics-ready Markdown conditioning
---

In [ ]:
# | default_exp preprocessing.conditioning.markdown

# Semantics-ready Markdown conditioning

Structured OCR preserves valuable page and region provenance, but its raw
Markdown is not automatically a sound semantic document. Running matter can
become content, detected titles can be flattened to one level, page boundaries
can split sentences and table rows, and table crops can duplicate recognized
table text. This notebook performs a deterministic, local conditioning pass
before `analyze_one_document_async` is called.

The layout sidecar is required: labels, bounding boxes, page dimensions, and
region identities make the transformations auditable and keep the source
Markdown immutable.

In [ ]:
# | export
from __future__ import annotations

import html
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field, replace
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Mapping, Sequence

import pypandoc
from tqdm.auto import tqdm

from ribosome.preprocessing.ocr.audit_repair import audit_layout_ocr_file
from ribosome.preprocessing.ocr.layout_bundle import (
    BoundingBox,
    LayoutBundlePaths,
    LayoutBundleValidator,
    normalize_html_table,
    parse_markdown_regions,
)


_EXPLICIT_BOILERPLATE_LABELS = frozenset({"header", "footer", "number"})
_TITLE_LABELS = frozenset({"paragraph_title", "title", "doc_title"})
_STRUCTURAL_TITLE_LABELS = frozenset({"paragraph_title", "title"})
_TOC_TITLES = frozenset({"目录", "目 录"})
_ATX_RE = re.compile(r"^[ \t]{0,3}#{1,6}[ \t]+(?P<title>.*?)(?:[ \t]+#+)?[ \t]*$")
_MULTILEVEL_NUMBER_RE = re.compile(
    r"^\s*(?P<number>\d+(?:\s*\.\s*\d+)+)\s*(?P<title>\D.*?)\s*$"
)
_SINGLELEVEL_NUMBER_RE = re.compile(
    r"^\s*(?P<number>\d+)\s*(?:[.]\s*)?\s+(?P<title>\S.*?)\s*$"
)
_TOC_ENTRY_RE = re.compile(
    r"^\s*(?P<number>\d+(?:\s*\.\s*\d+)*)\s+"
    r"(?P<title>.+?)(?:\s*\.{2,}\s*|\s+)"
    r"(?P<page>\d+|[IVXLCDM]+)\s*$",
    re.IGNORECASE,
)
_IMAGE_RE = re.compile(r"!\[[^\]]*\]\(<(?P<path>[^>]+)>\)")
_TABLE_IMAGE_LINE_RE = re.compile(r"(?m)^!\[[^\]]*Table[^\]]*\]\(<[^>]+>\)\s*$", re.IGNORECASE)
_WHITESPACE_RE = re.compile(r"\s+")
_CJK_RE = re.compile(r"[\u3400-\u9fff\uf900-\ufaff]")
_TERMINAL_RE = re.compile(r"[。！？.!?；;:]\s*$")
_LIST_START_RE = re.compile(r"^\s*(?:[-+*]\s+|\d+[.)、]\s*)")
_BLOCK_LINE_RE = re.compile(
    r"^\s*(?:#{1,6}\s+|[-+*]\s+|\d+[.)、]\s+|\||```|~~~|<table\b|!\[|\\\[|\$\$)",
    re.IGNORECASE,
)


@dataclass(frozen=True)
class MarkdownConditioningConfig:
    """Behavior and safety limits for structured Markdown conditioning."""

    running_text_page_ratio: float = 0.20
    page_margin_ratio: float = 0.12
    table_bottom_ratio: float = 0.78
    table_top_ratio: float = 0.22
    heading_title_similarity: float = 0.55
    max_table_rows: int = 12
    max_table_characters: int = 2600
    strip_heading_numbers: bool = True
    unnumbered_title_style: str = "bold"
    preserve_provenance: bool = True
    strict: bool = True

    def __post_init__(self) -> None:
        for name in (
            "running_text_page_ratio",
            "page_margin_ratio",
            "table_bottom_ratio",
            "table_top_ratio",
            "heading_title_similarity",
        ):
            value = getattr(self, name)
            if not 0 < value < 1:
                raise ValueError(f"{name} must be between 0 and 1")
        if self.max_table_rows < 2:
            raise ValueError("max_table_rows must be at least 2")
        if self.max_table_characters < 200:
            raise ValueError("max_table_characters must be at least 200")
        if self.unnumbered_title_style not in {"bold", "plain"}:
            raise ValueError("unnumbered_title_style must be 'bold' or 'plain'")


@dataclass
class ConditioningReport:
    """Auditable counts and warnings produced by one conditioning pass."""

    pages_total: int = 0
    input_regions: int = 0
    removed_label_regions: int = 0
    removed_repeated_regions: int = 0
    removed_running_title_regions: int = 0
    removed_boundary_lines: int = 0
    reordered_regions: int = 0
    joined_page_fragments: int = 0
    input_table_fragments: int = 0
    table_continuation_links: int = 0
    logical_table_sequences: int = 0
    output_table_chunks: int = 0
    table_fragments_accounted: int = 0
    headings_by_level: dict[int, int] = field(default_factory=dict)
    headings_total: int = 0
    demoted_titles: int = 0
    retained_images: int = 0
    dropped_table_images: int = 0
    dropped_nested_images: int = 0
    dropped_nested_regions: int = 0
    eligible: bool = False
    warnings: tuple[str, ...] = ()
    validation_errors: tuple[str, ...] = ()


@dataclass(frozen=True)
class SemanticsEligibilityReport:
    """Static Pandoc and asset checks for semantics-analysis readiness."""

    eligible: bool
    headings_by_level: dict[int, int]
    heading_titles: tuple[str, ...]
    table_count: int
    image_count: int
    top_level_sections: int
    errors: tuple[str, ...] = ()
    warnings: tuple[str, ...] = ()


@dataclass(frozen=True)
class ConditioningResult:
    """Conditioned Markdown together with its report and optional output path."""

    markdown: str
    report: ConditioningReport
    output_path: Path | None = None


@dataclass(frozen=True)
class _SourceRegion:
    page_number: int
    page_width: int
    page_height: int
    region_index: int
    label: str
    bbox: BoundingBox
    task_type: str
    status: str
    content: str
    payload: str
    asset: str | None

    @property
    def key(self) -> tuple[int, int]:
        return self.page_number, self.region_index

    @property
    def normalized_top(self) -> float:
        return self.bbox.y1 / self.page_height

    @property
    def normalized_bottom(self) -> float:
        return self.bbox.y2 / self.page_height


@dataclass(frozen=True)
class _Heading:
    number: str
    title: str
    level: int

## Region, boilerplate, and heading reconstruction

In [ ]:
# | export
def _strip_atx(value: str) -> str:
    lines = value.strip().splitlines()
    if not lines:
        return ""
    match = _ATX_RE.fullmatch(lines[0])
    if match:
        lines[0] = match.group("title").strip()
    return "\n".join(lines).strip()


def _normalize_compare(value: str) -> str:
    value = unicodedata.normalize("NFKC", html.unescape(_strip_atx(value))).casefold()
    value = value.replace("i\\0", "io").replace("i/0", "io").replace("i0", "io")
    return "".join(character for character in value if character.isalnum())


def _normalize_line(value: str) -> str:
    value = unicodedata.normalize("NFKC", html.unescape(_strip_atx(value))).casefold()
    return "".join(character for character in value if character.isalnum())


def _normalize_section_number(value: str) -> str:
    return ".".join(part.strip() for part in value.split(".") if part.strip())


def _parse_numbered_title(value: str) -> tuple[str, str] | None:
    plain = _WHITESPACE_RE.sub(" ", _strip_atx(value)).strip()
    match = _MULTILEVEL_NUMBER_RE.match(plain)
    if match is None:
        match = _SINGLELEVEL_NUMBER_RE.match(plain)
    if match is None:
        return None
    number = _normalize_section_number(match.group("number"))
    title = match.group("title").strip()
    if not number or not title:
        return None
    return number, title


def _title_similarity(first: str, second: str) -> float:
    left, right = _normalize_compare(first), _normalize_compare(second)
    if not left or not right:
        return 0.0
    if left == right:
        return 1.0
    if left in right or right in left:
        return min(len(left), len(right)) / max(len(left), len(right))
    return SequenceMatcher(None, left, right, autojunk=False).ratio()


def _layout_regions(markdown: str, layout: Mapping[str, Any], *, strict: bool) -> list[_SourceRegion]:
    marker_map = {marker.key: marker for marker in parse_markdown_regions(markdown)}
    regions: list[_SourceRegion] = []
    expected_keys: set[tuple[int, int]] = set()
    for page in layout.get("pages") or []:
        page_number = int(page["page_number"])
        width, height = int(page["width"]), int(page["height"])
        for record in page.get("regions") or []:
            index = int(record["index"])
            key = page_number, index
            expected_keys.add(key)
            marker = marker_map.get(key)
            if marker is None:
                if strict:
                    raise ValueError(f"layout region p{page_number}r{index} has no Markdown marker")
                continue
            bbox = BoundingBox.from_value(record.get("bbox"))
            if bbox is None:
                if strict:
                    raise ValueError(f"layout region p{page_number}r{index} has an invalid bbox")
                bbox = marker.bbox
            label = str(record.get("label") or "unknown")
            status = str(record.get("status") or "missing")
            if strict and (label != marker.label or status != marker.status or bbox != marker.bbox):
                raise ValueError(f"layout/Markdown marker mismatch at p{page_number}r{index}")
            regions.append(
                _SourceRegion(
                    page_number=page_number,
                    page_width=width,
                    page_height=height,
                    region_index=index,
                    label=label,
                    bbox=bbox,
                    task_type=str(record.get("task_type") or "text"),
                    status=status,
                    content=str(record.get("content") or "").strip(),
                    payload=marker.payload.strip(),
                    asset=str(record["asset"]) if record.get("asset") else None,
                )
            )
    if strict and set(marker_map) != expected_keys:
        extras = sorted(set(marker_map) - expected_keys)[:5]
        raise ValueError(f"Markdown contains markers absent from the layout sidecar: {extras}")
    return regions


def _repeated_margin_texts(
    regions: Sequence[_SourceRegion],
    pages_total: int,
    config: MarkdownConditioningConfig,
) -> set[str]:
    by_text_pages: dict[str, set[int]] = defaultdict(set)
    for region in regions:
        normalized = _normalize_compare(region.content or region.payload)
        at_margin = (
            region.normalized_top <= config.page_margin_ratio
            or region.normalized_bottom >= 1 - config.page_margin_ratio
        )
        if normalized and at_margin:
            by_text_pages[normalized].add(region.page_number)
    threshold = max(3, math.ceil(pages_total * config.running_text_page_ratio))
    return {text for text, pages in by_text_pages.items() if len(pages) >= threshold}


def _running_title_texts(
    regions: Sequence[_SourceRegion],
    pages_total: int,
    config: MarkdownConditioningConfig,
) -> set[str]:
    by_text_pages: dict[str, set[int]] = defaultdict(set)
    title_labels = _TITLE_LABELS | {"header", "figure_title"}
    for region in regions:
        if region.label not in title_labels or region.normalized_top > config.page_margin_ratio:
            continue
        normalized = _normalize_compare(region.content or region.payload)
        if normalized:
            by_text_pages[normalized].add(region.page_number)
    threshold = max(3, math.ceil(pages_total * config.running_text_page_ratio))
    return {text for text, pages in by_text_pages.items() if len(pages) >= threshold}


def _boundary_lexicon(regions: Sequence[_SourceRegion]) -> set[str]:
    lexicon: set[str] = set()
    for region in regions:
        if region.label not in _EXPLICIT_BOILERPLATE_LABELS:
            continue
        for line in (region.content or region.payload).splitlines():
            normalized = _normalize_line(line)
            if normalized:
                lexicon.add(normalized)
    return lexicon


def _document_code_parts(layout: Mapping[str, Any]) -> tuple[str, str]:
    source = str(layout.get("source") or "")
    code = re.search(r"([A-Z]{2})(\d{6})", source, flags=re.IGNORECASE)
    revision = re.search(r"\(([A-Z])[-/]?(\d+)\)", source, flags=re.IGNORECASE)
    numeric_code = code.group(2) if code else ""
    revision_code = f"{revision.group(1)}{revision.group(2)}".casefold() if revision else ""
    return numeric_code, revision_code


def _is_boundary_boilerplate(
    line: str,
    lexicon: set[str],
    *,
    numeric_code: str,
    revision_code: str,
) -> bool:
    normalized = _normalize_line(line)
    if not normalized:
        return True
    if normalized in lexicon:
        return True
    return bool(
        numeric_code
        and revision_code
        and numeric_code in normalized
        and normalized.endswith(revision_code)
        and len(normalized) <= len(numeric_code) + len(revision_code) + 3
    )


def _clean_boundary_lines(
    value: str,
    lexicon: set[str],
    *,
    numeric_code: str,
    revision_code: str,
) -> tuple[str, int]:
    lines = value.strip().splitlines()
    removed = 0
    while lines and _is_boundary_boilerplate(
        lines[0], lexicon, numeric_code=numeric_code, revision_code=revision_code
    ):
        lines.pop(0)
        removed += 1
    while lines:
        last_is_boilerplate = _is_boundary_boilerplate(
            lines[-1], lexicon, numeric_code=numeric_code, revision_code=revision_code
        )
        numeric_page_tail = bool(
            re.fullmatch(r"\s*\d(?:[ \t]*\d){0,3}\s*", lines[-1])
            and any(
                _is_boundary_boilerplate(
                    candidate,
                    lexicon,
                    numeric_code=numeric_code,
                    revision_code=revision_code,
                )
                for candidate in lines[-3:-1]
            )
        )
        if not (last_is_boilerplate or numeric_page_tail):
            break
        lines.pop()
        removed += 1
    return "\n".join(lines).strip(), removed


def _extract_toc(
    ordered_regions: Sequence[_SourceRegion],
) -> tuple[dict[str, str], int | None, int | None]:
    toc_start_position: int | None = None
    for position, region in enumerate(ordered_regions):
        if _strip_atx(region.content or region.payload).strip() in _TOC_TITLES:
            toc_start_position = position
            break
    if toc_start_position is None:
        return {}, None, None

    toc: dict[str, str] = {}
    body_start_position: int | None = None
    for position in range(toc_start_position + 1, len(ordered_regions)):
        region = ordered_regions[position]
        numbered = _parse_numbered_title(region.content or region.payload)
        if region.label in _STRUCTURAL_TITLE_LABELS and numbered is not None:
            body_start_position = position
            break
        for line in (region.content or region.payload).splitlines():
            match = _TOC_ENTRY_RE.match(_strip_atx(line))
            if match:
                toc[_normalize_section_number(match.group("number"))] = match.group("title").strip(" .…")
    return toc, toc_start_position, body_start_position


def _classify_headings(
    ordered_regions: Sequence[_SourceRegion],
    toc: Mapping[str, str],
    body_start_position: int | None,
    config: MarkdownConditioningConfig,
) -> dict[tuple[int, int], _Heading]:
    accepted: dict[tuple[int, int], _Heading] = {}
    accepted_numbers: set[str] = set()
    active: dict[int, str] = {}
    fallback_next_chapter = 1
    start = body_start_position if body_start_position is not None else 0
    for position, region in enumerate(ordered_regions):
        if position < start or region.label not in _STRUCTURAL_TITLE_LABELS:
            continue
        parsed = _parse_numbered_title(region.content or region.payload)
        if parsed is None:
            continue
        number, title = parsed
        level = number.count(".") + 1
        is_known = number in toc
        title_matches = is_known and _title_similarity(title, toc[number]) >= config.heading_title_similarity
        parent = number.rpartition(".")[0]
        active_parent = active.get(level - 1) if level > 1 else None
        inferred_child = (
            not is_known
            and level > 1
            and parent in accepted_numbers
            and active_parent == parent
        )
        fallback_chapter = (
            not toc
            and level == 1
            and int(number) == fallback_next_chapter
            and not title.rstrip().endswith((":", "："))
        )
        if not (title_matches or inferred_child or fallback_chapter):
            continue
        if number in accepted_numbers:
            continue
        accepted[region.key] = _Heading(number=number, title=title, level=level)
        accepted_numbers.add(number)
        for stale in [depth for depth in active if depth >= level]:
            active.pop(stale, None)
        active[level] = number
        if level == 1:
            fallback_next_chapter = int(number) + 1
    return accepted


def _stable_geometry_order(regions: Sequence[_SourceRegion]) -> tuple[list[_SourceRegion], int]:
    """Move only vertically disjoint inversions; retain overlap/index ordering."""
    ordered: list[_SourceRegion] = []
    moved: set[tuple[int, int]] = set()
    for region in regions:
        insertion = len(ordered)
        while insertion > 0:
            previous = ordered[insertion - 1]
            if region.bbox.y2 + 4 < previous.bbox.y1:
                insertion -= 1
            else:
                break
        if insertion != len(ordered):
            moved.add(region.key)
            moved.update(item.key for item in ordered[insertion:])
        ordered.insert(insertion, region)
    return ordered, len(moved)


def _is_nested_table_region(region: _SourceRegion, page_tables: Sequence[_SourceRegion]) -> bool:
    return region.task_type != "table" and any(
        table.key != region.key and table.bbox.contains(region.bbox, tolerance=4)
        for table in page_tables
    )

## Text continuity and table reconstruction

In [ ]:
# | export
def _join_inline(left: str, right: str) -> str:
    left, right = left.rstrip(), right.lstrip()
    if not left:
        return right
    if not right:
        return left
    if left.endswith("-") and left[-2:-1].isascii() and right[:1].isascii():
        return left[:-1] + right
    if _CJK_RE.fullmatch(left[-1]) or _CJK_RE.fullmatch(right[0]):
        separator = "" if not left.endswith((" ", "，", "、", "（", "(")) else ""
    elif left[-1].isalnum() and right[0].isalnum():
        separator = " "
    else:
        separator = ""
    return left + separator + right


def _reflow_plain_text(value: str) -> str:
    lines = value.strip().splitlines()
    if len(lines) <= 1 or any(not line.strip() for line in lines):
        return value.strip()
    if sum(bool(_TOC_ENTRY_RE.match(_strip_atx(line))) for line in lines) >= 2:
        return "\n\n".join(line.strip() for line in lines if line.strip())
    if any(_BLOCK_LINE_RE.match(line) for line in lines):
        return value.strip()
    result = lines[0].strip()
    for line in lines[1:]:
        result = _join_inline(result, line.strip())
    return result


def _can_join_page_prose(previous: _SourceRegion, following: _SourceRegion, left: str, right: str) -> bool:
    if previous.task_type != "text" or following.task_type != "text":
        return False
    allowed_labels = {"text", "content", "vision_footnote"}
    if previous.label not in allowed_labels or following.label not in allowed_labels:
        return False
    if previous.normalized_bottom < 0.86 or following.normalized_top > 0.18:
        return False
    if not left.strip() or not right.strip() or _TERMINAL_RE.search(left):
        return False
    if _LIST_START_RE.match(right) or _BLOCK_LINE_RE.match(right.splitlines()[0]):
        return False
    return True


def _row_signature(row: Sequence[str]) -> tuple[str, ...]:
    return tuple(_normalize_compare(cell) for cell in row)


def _pad_row(row: Sequence[str], width: int) -> list[str]:
    return [*row, *([""] * max(0, width - len(row)))]


def _merge_leading_continuation(rows: list[list[str]], incoming: list[list[str]]) -> bool:
    if not rows or not incoming:
        return False
    width = max(len(rows[-1]), len(incoming[0]))
    previous, first = _pad_row(rows[-1], width), _pad_row(incoming[0], width)
    nonempty = [index for index, value in enumerate(first) if value.strip()]
    if not nonempty or first[0].strip():
        return False
    for index in nonempty:
        previous[index] = _join_inline(previous[index], first[index])
    rows[-1] = previous
    incoming.pop(0)
    return True


def _escape_table_cell(value: str) -> str:
    value = html.unescape(str(value)).replace("\r", "").strip()
    value = value.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return value.replace("|", "\\|").replace("\n", "<br>")


def _split_rows(
    header: list[str],
    body: list[list[str]],
    config: MarkdownConditioningConfig,
) -> list[list[list[str]]]:
    groups: list[list[list[str]]] = []
    current: list[list[str]] = []
    current_size = 0
    body_limit = max(1, config.max_table_rows - 1)
    for row in body:
        row_size = len(" | ".join(row))
        if current and (
            len(current) >= body_limit
            or current_size + row_size > config.max_table_characters
        ):
            groups.append(current)
            current, current_size = [], 0
        current.append(row)
        current_size += row_size
    if current or not groups:
        groups.append(current)
    return [[header, *group] for group in groups]


def _render_markdown_table(rows: Sequence[Sequence[str]]) -> str:
    width = max((len(row) for row in rows), default=1)
    padded = [_pad_row(row, width) for row in rows]
    if not padded:
        padded = [["表格内容"]]
    header = padded[0]
    body = padded[1:] or [[""] * width]
    lines = [
        "| " + " | ".join(_escape_table_cell(cell) for cell in header) + " |",
        "| " + " | ".join("---" for _ in range(width)) + " |",
    ]
    lines.extend(
        "| " + " | ".join(_escape_table_cell(cell) for cell in row) + " |"
        for row in body
    )
    return "\n".join(lines)


def _plain_fragment_lines(value: str) -> list[str]:
    return [line.strip() for line in value.splitlines() if line.strip()]


def _fragment_grid(value: str) -> tuple[str, list[list[str]]]:
    cleaned = _TABLE_IMAGE_LINE_RE.sub("", value).strip()
    if "<table" in cleaned.casefold():
        normalized = normalize_html_table(cleaned)
        return "grid", [list(row) for row in normalized.rows if any(cell.strip() for cell in row)]
    lines = _plain_fragment_lines(cleaned)
    tabular = [line.split("\t") for line in lines]
    widths = Counter(len(row) for row in tabular if len(row) > 1)
    if widths:
        width, count = widths.most_common(1)[0]
        if count >= max(2, len(lines) // 2):
            return "grid", [_pad_row(row, width) for row in tabular]
    return "plain", [[line] for line in lines]


def _chunk_grid_rows(
    rows: list[list[str]],
    config: MarkdownConditioningConfig,
) -> list[str]:
    if not rows:
        return []
    width = max(len(row) for row in rows)
    padded = [_pad_row(row, width) for row in rows]
    header, body = padded[0], padded[1:]
    return [_render_markdown_table(group) for group in _split_rows(header, body, config)]


def _chunk_plain_rows(
    rows: list[list[str]],
    config: MarkdownConditioningConfig,
) -> list[str]:
    values = [row[0] for row in rows if row and row[0].strip()]
    if not values:
        return []
    groups: list[list[list[str]]] = []
    current: list[list[str]] = []
    current_size = 0
    for value in values:
        if current and (
            len(current) >= config.max_table_rows - 1
            or current_size + len(value) > config.max_table_characters
        ):
            groups.append(current)
            current, current_size = [], 0
        current.append([value])
        current_size += len(value)
    if current:
        groups.append(current)
    return [_render_markdown_table([["表格内容"], *group]) for group in groups]


def _table_assets(regions: Sequence[_SourceRegion]) -> tuple[str, ...]:
    return tuple(dict.fromkeys(region.asset for region in regions if region.asset))


def _render_table_component(
    regions: Sequence[_SourceRegion],
    logical_id: str,
    config: MarkdownConditioningConfig,
) -> list[str]:
    segments: list[tuple[str, list[list[str]]]] = []
    current_kind: str | None = None
    current_rows: list[list[str]] = []
    global_headers: dict[int, list[str]] = {}

    def flush() -> None:
        nonlocal current_kind, current_rows
        if current_kind and current_rows:
            segments.append((current_kind, current_rows))
        current_kind, current_rows = None, []

    for region in regions:
        kind, incoming = _fragment_grid(region.content or region.payload)
        if not incoming:
            continue
        if kind == "grid":
            width = max(len(row) for row in incoming)
            incoming = [_pad_row(row, width) for row in incoming]
            header = global_headers.get(width)
            if header is None:
                global_headers[width] = list(incoming[0])
                header = global_headers[width]
            elif incoming and _row_signature(incoming[0]) == _row_signature(header):
                incoming.pop(0)
            compatible = current_kind == "grid" and current_rows and max(len(row) for row in current_rows) == width
            if not compatible:
                flush()
                current_kind = "grid"
                current_rows = [list(header)]
            _merge_leading_continuation(current_rows, incoming)
            current_rows.extend(incoming)
        else:
            values = [row[0] for row in incoming if row and row[0].strip()]
            if current_kind != "plain":
                flush()
                current_kind = "plain"
            if current_rows and values:
                previous, following = current_rows[-1][0], values[0]
                if (
                    not _TERMINAL_RE.search(previous)
                    and not _LIST_START_RE.match(following)
                    and _normalize_compare(previous) != _normalize_compare(following)
                ):
                    current_rows[-1][0] = _join_inline(previous, following)
                    values.pop(0)
            current_rows.extend([[value] for value in values])
    flush()

    chunks: list[str] = []
    for kind, rows in segments:
        chunks.extend(
            _chunk_grid_rows(rows, config)
            if kind == "grid"
            else _chunk_plain_rows(rows, config)
        )
    if not chunks:
        chunks = [_render_markdown_table([["表格内容"], ["（无可用表格文本）"]])]

    first, last = regions[0], regions[-1]
    pages = f"{first.page_number}" if first.page_number == last.page_number else f"{first.page_number}-{last.page_number}"
    assets = "|".join(_table_assets(regions))
    rendered: list[str] = []
    for index, chunk in enumerate(chunks, start=1):
        prefix: list[str] = []
        if config.preserve_provenance:
            prefix.append(
                f"<!-- logical-table={logical_id} pages={pages} fragments={len(regions)} "
                f"chunk={index}/{len(chunks)} assets={json.dumps(assets, ensure_ascii=False)} -->"
            )
        if index > 1:
            prefix.append(f"**续表 {index}/{len(chunks)}**")
        prefix.append(chunk)
        rendered.append("\n\n".join(prefix))
    return rendered


class _UnionFind:
    def __init__(self, keys: Sequence[tuple[int, int]]):
        self.parent = {key: key for key in keys}

    def find(self, key: tuple[int, int]) -> tuple[int, int]:
        parent = self.parent[key]
        if parent != key:
            self.parent[key] = self.find(parent)
        return self.parent[key]

    def union(self, first: tuple[int, int], second: tuple[int, int]) -> None:
        left, right = self.find(first), self.find(second)
        if left != right:
            self.parent[right] = left


def _table_components(
    pages: Mapping[int, Sequence[_SourceRegion]],
    config: MarkdownConditioningConfig,
) -> tuple[dict[tuple[int, int], list[_SourceRegion]], int]:
    tables = [region for regions in pages.values() for region in regions if region.task_type == "table"]
    union = _UnionFind([region.key for region in tables])
    links = 0
    for page_number in range(1, max(pages, default=0)):
        previous_page = list(pages.get(page_number, ()))
        following_page = list(pages.get(page_number + 1, ()))
        if not previous_page or not following_page:
            continue
        previous, following = previous_page[-1], following_page[0]
        if (
            previous.task_type == "table"
            and following.task_type == "table"
            and previous.normalized_bottom >= config.table_bottom_ratio
            and following.normalized_top <= config.table_top_ratio
        ):
            union.union(previous.key, following.key)
            links += 1
    grouped: dict[tuple[int, int], list[_SourceRegion]] = defaultdict(list)
    for region in tables:
        grouped[union.find(region.key)].append(region)
    components: dict[tuple[int, int], list[_SourceRegion]] = {}
    for members in grouped.values():
        ordered = sorted(members, key=lambda region: (region.page_number, region.region_index))
        components[ordered[0].key] = ordered
    return components, links

## Conditioning orchestration and validation

In [ ]:
# | export
def _format_unnumbered_title(value: str, config: MarkdownConditioningConfig) -> str:
    plain = _strip_atx(value).strip()
    if not plain:
        return ""
    if config.unnumbered_title_style == "plain":
        return plain
    return f"**{plain.replace(chr(10), '<br>')}**"


def _render_heading(region: _SourceRegion, heading: _Heading, config: MarkdownConditioningConfig) -> str:
    title = heading.title if config.strip_heading_numbers else f"{heading.number} {heading.title}"
    parts = []
    if config.preserve_provenance:
        parts.append(
            f"<!-- section-number={heading.number} source-page={region.page_number} "
            f"source-region={region.region_index} -->"
        )
    parts.append(f"{'#' * heading.level} {title}")
    return "\n\n".join(parts)


def _render_region_payload(
    region: _SourceRegion,
    cleaned_payload: str,
    heading: _Heading | None,
    config: MarkdownConditioningConfig,
) -> str:
    if heading is not None:
        return _render_heading(region, heading, config)
    if region.label in _TITLE_LABELS:
        return _format_unnumbered_title(cleaned_payload, config)
    if region.task_type == "figure":
        parts = []
        if config.preserve_provenance:
            parts.append(
                f"<!-- figure source-page={region.page_number} source-region={region.region_index} "
                f"asset={json.dumps(region.asset or '', ensure_ascii=False)} -->"
            )
        parts.append(cleaned_payload)
        return "\n\n".join(part for part in parts if part)
    return _reflow_plain_text(cleaned_payload)


def condition_layout_markdown(
    markdown: str,
    layout: Mapping[str, Any],
    *,
    config: MarkdownConditioningConfig | None = None,
) -> ConditioningResult:
    """Condition marker-delimited OCR Markdown without writing any files."""
    selected = config or MarkdownConditioningConfig()
    pages_total = int(layout.get("pages_total") or len(layout.get("pages") or []))
    if selected.strict and int(layout.get("schema_version") or 0) != 2:
        raise ValueError("conditioning requires a schema-v2 layout sidecar")
    regions = _layout_regions(markdown, layout, strict=selected.strict)
    report = ConditioningReport(
        pages_total=pages_total,
        input_regions=len(regions),
        input_table_fragments=sum(region.task_type == "table" for region in regions),
        dropped_table_images=sum(region.task_type == "table" and bool(region.asset) for region in regions),
    )
    repeated = _repeated_margin_texts(regions, pages_total, selected)
    running_titles = _running_title_texts(regions, pages_total, selected)
    lexicon = _boundary_lexicon(regions)
    numeric_code, revision_code = _document_code_parts(layout)

    page_candidates: dict[int, list[_SourceRegion]] = defaultdict(list)
    payloads: dict[tuple[int, int], str] = {}
    for region in regions:
        normalized = _normalize_compare(region.content or region.payload)
        if normalized and normalized in running_titles:
            report.removed_running_title_regions += 1
            report.removed_repeated_regions += 1
            continue
        if region.label in _EXPLICIT_BOILERPLATE_LABELS:
            report.removed_label_regions += 1
            continue
        if normalized and normalized in repeated:
            report.removed_repeated_regions += 1
            continue
        value = region.content if region.task_type == "table" else region.payload
        value, removed_lines = _clean_boundary_lines(
            value,
            lexicon,
            numeric_code=numeric_code,
            revision_code=revision_code,
        )
        report.removed_boundary_lines += removed_lines
        payloads[region.key] = value
        page_candidates[region.page_number].append(region)

    semantic_pages: dict[int, list[_SourceRegion]] = {}
    for page_number in range(1, pages_total + 1):
        candidates = page_candidates.get(page_number, [])
        page_tables = [region for region in candidates if region.task_type == "table"]
        filtered = []
        for region in candidates:
            if _is_nested_table_region(region, page_tables):
                report.dropped_nested_regions += 1
                if region.task_type == "figure":
                    report.dropped_nested_images += 1
                continue
            if not payloads.get(region.key) and region.task_type != "figure":
                continue
            filtered.append(region)
        ordered, moved = _stable_geometry_order(filtered)
        report.reordered_regions += moved
        semantic_pages[page_number] = ordered

    ordered_regions = [region for page in semantic_pages.values() for region in page]
    toc, _, body_start = _extract_toc(ordered_regions)
    headings = _classify_headings(ordered_regions, toc, body_start, selected)
    heading_counts = Counter(heading.level for heading in headings.values())
    report.headings_by_level = dict(sorted(heading_counts.items()))
    report.headings_total = sum(heading_counts.values())
    report.demoted_titles = sum(
        region.label in _TITLE_LABELS and region.key not in headings
        for region in ordered_regions
    )

    table_components, continuation_links = _table_components(semantic_pages, selected)
    table_member_to_first: dict[tuple[int, int], tuple[int, int]] = {}
    for first, members in table_components.items():
        for member in members:
            table_member_to_first[member.key] = first
    report.table_continuation_links = continuation_links
    report.logical_table_sequences = sum(len(members) > 1 for members in table_components.values())
    report.table_fragments_accounted = sum(len(members) for members in table_components.values())

    blocks: list[tuple[str, _SourceRegion, _SourceRegion, str]] = []
    table_sequence = 0
    for region in ordered_regions:
        if region.task_type == "table":
            first_key = table_member_to_first[region.key]
            if region.key != first_key:
                continue
            table_sequence += 1
            members = table_components[first_key]
            chunks = _render_table_component(members, f"table-{table_sequence:04d}", selected)
            report.output_table_chunks += len(chunks)
            blocks.extend((chunk, members[0], members[-1], "table") for chunk in chunks)
            continue

        value = payloads.get(region.key, "")
        rendered = _render_region_payload(region, value, headings.get(region.key), selected)
        if not rendered:
            continue
        if (
            blocks
            and blocks[-1][3] == "prose"
            and blocks[-1][2].page_number + 1 == region.page_number
            and _can_join_page_prose(blocks[-1][2], region, blocks[-1][0], rendered)
        ):
            previous_text, first_region, _, kind = blocks[-1]
            blocks[-1] = (_join_inline(previous_text, rendered), first_region, region, kind)
            report.joined_page_fragments += 1
        else:
            kind = "prose" if region.task_type == "text" and region.label in {"text", "content", "vision_footnote"} else "other"
            blocks.append((rendered, region, region, kind))

    output = "\n\n".join(block[0].strip() for block in blocks if block[0].strip()).strip() + "\n"
    report.retained_images = len(_IMAGE_RE.findall(output))
    eligibility = validate_semantics_ready_markdown(
        output,
        expected_heading_counts=report.headings_by_level,
        require_numberless_headings=selected.strip_heading_numbers,
    )
    report.eligible = eligibility.eligible
    report.validation_errors = eligibility.errors
    report.warnings = eligibility.warnings
    if selected.strict and not eligibility.eligible:
        raise ValueError("conditioned Markdown failed eligibility checks: " + "; ".join(eligibility.errors))
    return ConditioningResult(markdown=output, report=report)


def _walk_pandoc_nodes(value: Any):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from _walk_pandoc_nodes(child)
    elif isinstance(value, list):
        for child in value:
            yield from _walk_pandoc_nodes(child)


def _pandoc_inline_text(value: Any) -> str:
    if isinstance(value, dict):
        if value.get("t") == "Str":
            return str(value.get("c") or "")
        if value.get("t") in {"Space", "SoftBreak", "LineBreak"}:
            return " "
        return _pandoc_inline_text(value.get("c"))
    if isinstance(value, list):
        return "".join(_pandoc_inline_text(item) for item in value)
    return ""


def validate_semantics_ready_markdown(
    markdown: str,
    *,
    source_dir: str | Path | None = None,
    expected_heading_counts: Mapping[int, int] | None = None,
    require_numberless_headings: bool = True,
) -> SemanticsEligibilityReport:
    """Parse conditioned Markdown and check its section tree, tables, and assets."""
    errors: list[str] = []
    warnings: list[str] = []
    if re.search(r"<\/?table\b", markdown, flags=re.IGNORECASE):
        errors.append("raw HTML table markup remains")
    try:
        ast = json.loads(pypandoc.convert_text(markdown, "json", format="md"))
    except Exception as error:
        return SemanticsEligibilityReport(
            False, {}, (), 0, 0, 0, (f"Pandoc parse failed: {type(error).__name__}: {error}",), ()
        )
    headings: list[tuple[int, str]] = []
    table_count = 0
    image_count = 0
    for node in _walk_pandoc_nodes(ast):
        node_type = node.get("t")
        if node_type == "Header":
            level = int(node["c"][0])
            headings.append((level, _WHITESPACE_RE.sub(" ", _pandoc_inline_text(node["c"][2])).strip()))
        elif node_type == "Table":
            table_count += 1
        elif node_type == "Image":
            image_count += 1
    counts = dict(sorted(Counter(level for level, _ in headings).items()))
    if expected_heading_counts is not None and counts != dict(expected_heading_counts):
        errors.append(f"heading levels are {counts}, expected {dict(expected_heading_counts)}")
    previous_level = 0
    for level, title in headings:
        if previous_level and level > previous_level + 1:
            errors.append(f"heading level jumps from {previous_level} to {level} at {title!r}")
        previous_level = level
        if require_numberless_headings and _parse_numbered_title(title) is not None:
            errors.append(f"visible section number remains in heading {title!r}")
    if headings and headings[0][0] != 1:
        errors.append("the first semantic heading is not level one")
    if source_dir is not None:
        root = Path(source_dir)
        for match in _IMAGE_RE.finditer(markdown):
            target = match.group("path")
            if re.match(r"^[a-z]+://", target, flags=re.IGNORECASE):
                continue
            if not (root / target).is_file():
                errors.append(f"missing local image: {target}")
    return SemanticsEligibilityReport(
        eligible=not errors,
        headings_by_level=counts,
        heading_titles=tuple(title for _, title in headings),
        table_count=table_count,
        image_count=image_count,
        top_level_sections=counts.get(1, 0),
        errors=tuple(dict.fromkeys(errors)),
        warnings=tuple(dict.fromkeys(warnings)),
    )


def condition_markdown_file(
    source: str | Path,
    *,
    layout_path: str | Path | None = None,
    output_path: str | Path | None = None,
    overwrite: bool = False,
    config: MarkdownConditioningConfig | None = None,
) -> ConditioningResult:
    """Validate, condition, and write a sibling Markdown file without mutating its source."""
    source_path = Path(source).expanduser().resolve()
    if not source_path.is_file():
        raise FileNotFoundError(source_path)
    sidecar = (
        Path(layout_path).expanduser().resolve()
        if layout_path is not None
        else source_path.with_suffix(".layout.json")
    )
    if not sidecar.is_file():
        raise FileNotFoundError(sidecar)
    target = (
        Path(output_path).expanduser().resolve()
        if output_path is not None
        else source_path.with_name(f"{source_path.stem}.conditioned.md")
    )
    if target == source_path:
        raise ValueError("output_path must not overwrite the source Markdown")
    if target.exists() and not overwrite:
        raise FileExistsError(f"conditioned output already exists: {target}")

    selected = config or MarkdownConditioningConfig()
    paths = LayoutBundlePaths.from_layout(sidecar, markdown_path=source_path)
    LayoutBundleValidator().validate(paths, strict=selected.strict)
    audit = audit_layout_ocr_file(sidecar)
    if selected.strict and audit.needs_repair:
        details = [*audit.file_reasons]
        details.extend(
            f"p{issue.page_number}r{issue.region_index}: {'; '.join(issue.reasons)}"
            for issue in audit.region_issues
        )
        raise ValueError("layout OCR audit has unresolved findings: " + " | ".join(details))

    layout = json.loads(sidecar.read_text(encoding="utf-8"))
    conditioned = condition_layout_markdown(
        source_path.read_text(encoding="utf-8"),
        layout,
        config=selected,
    )
    asset_check = validate_semantics_ready_markdown(
        conditioned.markdown,
        source_dir=source_path.parent,
        expected_heading_counts=conditioned.report.headings_by_level,
        require_numberless_headings=selected.strip_heading_numbers,
    )
    conditioned.report.eligible = asset_check.eligible
    conditioned.report.validation_errors = asset_check.errors
    conditioned.report.warnings = asset_check.warnings
    if selected.strict and not asset_check.eligible:
        raise ValueError("conditioned Markdown failed file checks: " + "; ".join(asset_check.errors))
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(conditioned.markdown, encoding="utf-8")
    return replace(conditioned, output_path=target)

## Focused regression tests

In [ ]:
# | hide
from tempfile import TemporaryDirectory

from fastcore.test import test_eq, test_fail


def _marker(page: int, index: int, label: str, bbox: tuple[int, int, int, int], payload: str) -> str:
    return (
        f"<!-- Page {page} -->\n\n"
        f"<!-- layout-region page={page} index={index} label={label} "
        f"bbox={','.join(map(str, bbox))} status=completed -->\n\n{payload}\n"
    )


def _layout_region(index: int, label: str, bbox: tuple[int, int, int, int], content: str, task: str = "text") -> dict[str, Any]:
    return {
        "index": index,
        "label": label,
        "bbox": list(bbox),
        "task_type": task,
        "status": "completed",
        "content": content,
        "raw_content": content,
        "asset": None,
    }


def _synthetic_layout(pages: list[list[dict[str, Any]]]) -> dict[str, Any]:
    return {
        "schema_version": 2,
        "pages_total": len(pages),
        "status": "processed",
        "source": "SX000001《测试手册》(A-1).pdf",
        "pages": [
            {
                "page_number": number,
                "width": 1000,
                "height": 1000,
                "status": "completed",
                "regions": regions,
            }
            for number, regions in enumerate(pages, start=1)
        ],
    }


def test_heading_and_boilerplate_conditioning():
    pages = [
        [
            _layout_region(1, "paragraph_title", (50, 20, 600, 60), "测试手册"),
            _layout_region(2, "paragraph_title", (50, 100, 150, 140), "目录"),
            _layout_region(3, "content", (50, 160, 900, 800), "1 第一章 1\n1.1 小节 2\nSX000001-A/1\n测试公司\n1"),
            _layout_region(4, "footer", (50, 930, 300, 960), "SX000001-A/1"),
            _layout_region(5, "footer", (400, 930, 600, 960), "测试公司"),
            _layout_region(6, "number", (900, 930, 930, 960), "1"),
        ],
        [
            _layout_region(1, "paragraph_title", (50, 20, 600, 60), "测试手册"),
            _layout_region(2, "paragraph_title", (50, 100, 300, 140), "1 第一章"),
            _layout_region(3, "paragraph_title", (50, 200, 300, 240), "1.1 小节"),
            _layout_region(4, "paragraph_title", (50, 300, 400, 340), "2 点法步骤:"),
            _layout_region(5, "footer", (50, 930, 300, 960), "SX000001-A/1"),
            _layout_region(6, "footer", (400, 930, 600, 960), "测试公司"),
            _layout_region(7, "number", (900, 930, 930, 960), "2"),
        ],
        [
            _layout_region(1, "paragraph_title", (50, 20, 600, 60), "测试手册"),
            _layout_region(2, "text", (50, 100, 500, 140), "正文"),
            _layout_region(3, "footer", (50, 930, 300, 960), "SX000001-A/1"),
            _layout_region(4, "footer", (400, 930, 600, 960), "测试公司"),
            _layout_region(5, "number", (900, 930, 930, 960), "3"),
        ],
    ]
    layout = _synthetic_layout(pages)
    markdown = "".join(
        _marker(page_number, region["index"], region["label"], tuple(region["bbox"]),
                f"## {region['content']}" if region["label"] == "paragraph_title" else region["content"])
        for page_number, regions in enumerate(pages, start=1)
        for region in regions
    )
    result = condition_layout_markdown(markdown, layout)
    assert "# 第一章" in result.markdown
    assert "## 小节" in result.markdown
    assert "**2 点法步骤:**" in result.markdown
    assert "## 测试手册" not in result.markdown
    assert "SX000001-A/1" not in result.markdown
    test_eq(result.report.headings_by_level, {1: 1, 2: 1})


def test_geometry_and_cross_page_joining():
    first = _SourceRegion(1, 1000, 1000, 1, "text", BoundingBox(50, 900, 900, 960), "text", "completed", "机器人运", "机器人运", None)
    second = _SourceRegion(2, 1000, 1000, 1, "text", BoundingBox(50, 40, 900, 120), "text", "completed", "行正常。", "行正常。", None)
    assert _can_join_page_prose(first, second, first.payload, second.payload)
    test_eq(_join_inline(first.payload, second.payload), "机器人运行正常。")
    low = replace(first, region_index=2, bbox=BoundingBox(50, 300, 900, 340))
    high = replace(first, region_index=3, bbox=BoundingBox(50, 200, 900, 240))
    ordered, moved = _stable_geometry_order([low, high])
    test_eq([region.region_index for region in ordered], [3, 2])
    assert moved == 2


def test_table_continuation_and_chunking():
    rows = [["报警码", "解决措施"], ["1718", "配置了DI功能30且"]]
    incoming = [["", "输入有效"], ["1719", "重新启动"]]
    assert _merge_leading_continuation(rows, incoming)
    test_eq(rows[-1], ["1718", "配置了DI功能30且输入有效"])
    test_eq(incoming, [["1719", "重新启动"]])
    config = MarkdownConditioningConfig(max_table_rows=3, max_table_characters=300)
    chunks = _chunk_grid_rows([["A", "B"], ["1", "x"], ["2", "y"], ["3", "z"]], config)
    test_eq(len(chunks), 2)
    assert all("| A | B |" in chunk for chunk in chunks)


def test_overwrite_protection():
    with TemporaryDirectory() as directory:
        root = Path(directory)
        source = root / "manual.md"
        sidecar = root / "manual.layout.json"
        output = root / "manual.conditioned.md"
        source.write_text("source", encoding="utf-8")
        sidecar.write_text("{}", encoding="utf-8")
        output.write_text("existing", encoding="utf-8")
        test_fail(
            lambda: condition_markdown_file(source, output_path=output),
            contains="already exists",
        )


test_heading_and_boilerplate_conditioning()
test_geometry_and_cross_page_joining()
test_table_continuation_and_chunking()
test_overwrite_protection()

## Generate and validate SX322001

The output remains beside the source so all retained relative figure links
continue to resolve. The default refuses to overwrite an existing result.

In [ ]:
from dataclasses import asdict


PROJ_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").is_file()
)
SX322001_SOURCE = (
    PROJ_ROOT
    / "assets/PDF-20260721/.md_unlimited/03用户手册/软件类/V4.0手册"
    / "SX322001《新松机器人通用操作手册》(B-7).md"
)
SX322001_OUTPUT = SX322001_SOURCE.with_name(
    f"{SX322001_SOURCE.stem}.conditioned.md"
)

sx322001_result = condition_markdown_file(
    SX322001_SOURCE,
    output_path=SX322001_OUTPUT,
    overwrite=True,
)
sx322001_eligibility = validate_semantics_ready_markdown(
    sx322001_result.markdown,
    source_dir=SX322001_SOURCE.parent,
    expected_heading_counts={1: 10, 2: 78, 3: 125, 4: 13},
)

assert sx322001_result.report.headings_total == 226
assert sx322001_result.report.headings_by_level == {1: 10, 2: 78, 3: 125, 4: 13}
assert sx322001_result.report.input_table_fragments == 495
assert sx322001_result.report.table_fragments_accounted == 495
assert sx322001_result.report.logical_table_sequences == 53
assert sx322001_result.report.removed_running_title_regions == 726
assert "机器人停止运行的情况" in sx322001_result.markdown
assert "配置了DI功能30且输入有效" in sx322001_result.markdown
assert "# 5 点法标定步骤" not in sx322001_result.markdown
assert "# 3 点法示教" not in sx322001_result.markdown
assert not re.search(r"(?m)^# 坐标系指令$", sx322001_result.markdown)
assert "## 坐标系指令" in sx322001_result.markdown
assert "<table" not in sx322001_result.markdown.casefold()
assert "<!-- Page " not in sx322001_result.markdown
assert not re.search(r"(?m)^S?X?322001-B/7$", sx322001_result.markdown)
assert not re.search(r"(?m)^#{1,6} 新松机器人通用操作手册$", sx322001_result.markdown)
assert sx322001_result.markdown.index("机器人：") < sx322001_result.markdown.index("机器人本体机械部分")
assert sx322001_eligibility.eligible, sx322001_eligibility.errors
assert sx322001_eligibility.top_level_sections == 10
assert sx322001_eligibility.table_count == sx322001_result.report.output_table_chunks

from ribosome.core.dom.model import DOMClass


with TemporaryDirectory() as temporary_directory:
    smoke_path = Path(temporary_directory) / SX322001_OUTPUT.name
    smoke_path.write_text(sx322001_result.markdown, encoding="utf-8")
    smoke_dom = DOMClass(smoke_path)
    smoke_dom.setup()
    smoke_ast = json.loads(smoke_dom.ast_json)
    smoke_sections = [
        block
        for block in smoke_ast["blocks"]
        if isinstance(block, dict) and block.get("t") == "Section"
    ]
    assert len(smoke_sections) == 10
    assert all(section["c"][0]["c"][0] == 1 for section in smoke_sections)

asdict(sx322001_result.report)

## Recursively condition a folder

`condition_markdown_folder` discovers source Markdown files recursively,
requires each source to have a sibling `.layout.json`, and writes sibling
`.conditioned.md` outputs. Generated outputs are never treated as inputs.
Existing outputs are skipped unless `overwrite=True`; a failed document is
recorded without preventing the remaining documents from being processed.
A notebook-aware progress bar is enabled by default and can be disabled with
`show_progress=False`.

In [ ]:
# | export
@dataclass(frozen=True)
class FolderConditioningItem:
    """Outcome for one Markdown source discovered by a folder run."""

    source_path: Path
    output_path: Path
    status: str
    report: ConditioningReport | None = None
    error: str | None = None


@dataclass(frozen=True)
class FolderConditioningResult:
    """Complete, deterministic accounting for a recursive folder run."""

    root: Path
    items: tuple[FolderConditioningItem, ...]

    @property
    def status_counts(self) -> dict[str, int]:
        return dict(Counter(item.status for item in self.items))

    @property
    def failures(self) -> tuple[FolderConditioningItem, ...]:
        return tuple(item for item in self.items if item.status == "failed")


def condition_markdown_folder(
    source_root: str | Path,
    *,
    overwrite: bool = False,
    continue_on_error: bool = True,
    show_progress: bool = True,
    config: MarkdownConditioningConfig | None = None,
) -> FolderConditioningResult:
    """Condition every eligible Markdown file below ``source_root`` recursively."""
    root = Path(source_root).expanduser().resolve()
    if not root.is_dir():
        raise NotADirectoryError(root)

    sources = sorted(
        (
            path
            for path in root.rglob("*.md")
            if path.is_file() and not path.name.casefold().endswith(".conditioned.md")
        ),
        key=lambda path: path.relative_to(root).as_posix().casefold(),
    )
    items: list[FolderConditioningItem] = []
    progress = tqdm(
        sources,
        desc="Conditioning Markdown",
        unit="file",
        dynamic_ncols=True,
        disable=not show_progress,
    )
    for source in progress:
        progress.set_postfix_str(source.relative_to(root).as_posix(), refresh=False)
        layout_path = source.with_suffix(".layout.json")
        output_path = source.with_name(f"{source.stem}.conditioned.md")
        if not layout_path.is_file():
            items.append(FolderConditioningItem(source, output_path, "missing_layout"))
            continue
        if output_path.exists() and not overwrite:
            items.append(FolderConditioningItem(source, output_path, "skipped_existing"))
            continue
        try:
            result = condition_markdown_file(
                source,
                layout_path=layout_path,
                output_path=output_path,
                overwrite=overwrite,
                config=config,
            )
        except Exception as error:
            items.append(
                FolderConditioningItem(
                    source,
                    output_path,
                    "failed",
                    error=f"{type(error).__name__}: {error}",
                )
            )
            if not continue_on_error:
                progress.close()
                raise
        else:
            items.append(
                FolderConditioningItem(
                    source,
                    result.output_path or output_path,
                    "conditioned",
                    report=result.report,
                )
            )
    progress.close()
    return FolderConditioningResult(root=root, items=tuple(items))

In [ ]:
# | hide
def test_recursive_folder_conditioning_discovery():
    with TemporaryDirectory() as directory:
        root = Path(directory)
        nested = root / "nested"
        nested.mkdir()
        ready = nested / "ready.md"
        missing = nested / "missing.md"
        ready.write_text("source", encoding="utf-8")
        missing.write_text("source", encoding="utf-8")
        ready.with_suffix(".layout.json").write_text("{}", encoding="utf-8")
        ready.with_name("ready.conditioned.md").write_text("existing", encoding="utf-8")

        result = condition_markdown_folder(root, show_progress=False)

        test_eq([item.source_path.name for item in result.items], ["missing.md", "ready.md"])
        test_eq(result.status_counts, {"missing_layout": 1, "skipped_existing": 1})
        test_eq(result.failures, ())


test_recursive_folder_conditioning_discovery()

The following manual-run cell defaults to
`PROJ_ROOT/assets/unlimited_ocr_mit_conditioning`. Change `BATCH_SOURCE_ROOT`
to process another tree, or set `BATCH_OVERWRITE = True` to regenerate
existing outputs. It is excluded from automated notebook execution because
it intentionally processes the complete corpus.

In [ ]:
# | eval: false
BATCH_SOURCE_ROOT = PROJ_ROOT / "assets" / "unlimited_ocr_mit_conditioning"
BATCH_OVERWRITE = False
BATCH_SHOW_PROGRESS = True

folder_result = condition_markdown_folder(
    BATCH_SOURCE_ROOT,
    overwrite=BATCH_OVERWRITE,
    show_progress=BATCH_SHOW_PROGRESS,
)
folder_summary = {
    "root": str(folder_result.root),
    "discovered": len(folder_result.items),
    **folder_result.status_counts,
}
print(json.dumps(folder_summary, ensure_ascii=False, indent=2))
for item in folder_result.failures:
    print(f"FAILED {item.source_path.relative_to(folder_result.root)}: {item.error}")